In [1]:
import os
import warnings
from glob import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:

print("Loading cleaned parquet files...")

parquet_files = sorted(glob("./data/cleaned_partitions/cleaned_part*.parquet"))

df_list = [pd.read_parquet(file) for file in parquet_files]
main_df = pd.concat(df_list, ignore_index=True)

print(f"Dataset loaded successfully. Total rows: {len(main_df):,}")

print("\n--- Current Borough Counts ---")
print(main_df["Borough"].value_counts())

Loading cleaned parquet files...
Dataset loaded successfully. Total rows: 10,607,281

--- Current Borough Counts ---
Borough
Manhattan        9815679
Queens            603435
Brooklyn          179940
Bronx               8156
Staten Island         71
Name: count, dtype: int64


In [3]:
target_total_samples = 2000

# currently set to balanced 25p
borough_weights = {
    'Manhattan': 0.25,
    'Queens': 0.25,
    'Brooklyn': 0.25,
    'Bronx': 0.25
}

In [4]:
# Staten Island ones directly taken (because it only has 79 samples)

staten_island_samples = main_df[main_df['Borough'] == 'Staten Island']

sampled_pieces = [staten_island_samples]

for borough, weight in borough_weights.items():
    target_size = int(target_total_samples * weight)
    
    borough_data = main_df[main_df['Borough'] == borough]
    
    sampled_subset = borough_data.sample(n=target_size, random_state=42)
    
    sampled_pieces.append(sampled_subset)
    print(f"Sampled {target_size} rows for {borough} (Target was {weight*100}%).")

Sampled 500 rows for Manhattan (Target was 25.0%).
Sampled 500 rows for Queens (Target was 25.0%).
Sampled 500 rows for Brooklyn (Target was 25.0%).
Sampled 500 rows for Bronx (Target was 25.0%).


In [5]:
# combine everything
final_sampled_df = pd.concat(sampled_pieces, ignore_index=True)
print(f"\nFinal sampled dataset size: {len(final_sampled_df)} rows.")


Final sampled dataset size: 2071 rows.


In [6]:
print("\nApplying StandardScaler to clustering features...")

features_to_scale = [
    'Speed', 
    'hour_sin', 'hour_cos', 
    'day_sin', 'day_cos', 
    'log_trip_times', 'log_trip_distance', 'log_total_amount', 
    'passenger_count',
    'pickup_longitude', 'pickup_latitude',
    'dropoff_longitude', 'dropoff_latitude']

scaler = StandardScaler()

final_sampled_df[features_to_scale] = scaler.fit_transform(final_sampled_df[features_to_scale])


Applying StandardScaler to clustering features...


In [7]:
print("\n--- Verification: Post-Scaling Means ---")
print(final_sampled_df[features_to_scale].mean().round(3))
print("\n--- Verification: Post-Scaling STDs ---")
print(final_sampled_df[features_to_scale].std().round(3))


--- Verification: Post-Scaling Means ---
Speed                0.0
hour_sin            -0.0
hour_cos             0.0
day_sin              0.0
day_cos             -0.0
log_trip_times      -0.0
log_trip_distance   -0.0
log_total_amount    -0.0
passenger_count      0.0
pickup_longitude    -0.0
pickup_latitude      0.0
dropoff_longitude   -0.0
dropoff_latitude     0.0
dtype: float64

--- Verification: Post-Scaling STDs ---
Speed                1.0
hour_sin             1.0
hour_cos             1.0
day_sin              1.0
day_cos              1.0
log_trip_times       1.0
log_trip_distance    1.0
log_total_amount     1.0
passenger_count      1.0
pickup_longitude     1.0
pickup_latitude      1.0
dropoff_longitude    1.0
dropoff_latitude     1.0
dtype: float64


In [9]:
output_file = './data/proportional_sampled_2k.parquet'
final_sampled_df.to_parquet(output_file, index=False)
print(f"\nSuccess! Exported {output_file} ready for MDS and K-Means.")


Success! Exported ./data/proportional_sampled_2k.parquet ready for MDS and K-Means.
